# 01 - 探索 LIBERO 数据集

目标：搞清楚 LIBERO 的人类示教数据长什么样。

每跑完一格，看输出，理解之后再跑下一格。

我们要回答 4 个问题：
1. 每个 hdf5 文件里装了什么？
2. 一条示教（demo）有多少帧？
3. 每一帧的图像、动作、状态分别是什么形状？
4. 把它"放出来看"，机械臂在做什么？


In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from libero.libero import benchmark, get_libero_path

print("✅ 库导入成功")
print("LIBERO datasets path:", get_libero_path("datasets"))


## 1. 列出数据集文件

`libero_spatial` 共 10 个任务，每个任务对应一个 `.hdf5` 文件。


In [ ]:
datasets_root = Path(get_libero_path("datasets")).resolve()
spatial_dir = datasets_root / "libero_spatial"

hdf5_files = sorted(spatial_dir.glob("*.hdf5"))
print(f"找到 {len(hdf5_files)} 个任务文件:\n")
for i, f in enumerate(hdf5_files):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  [{i}] {f.name}  ({size_mb:.1f} MB)")


## 2. 打开第 0 个文件，看它的结构

HDF5 是个层级结构（像文件系统）。我们递归打印一下，看看里面有什么 group 和 dataset。


In [ ]:
demo_file = hdf5_files[0]
print(f"打开文件: {demo_file.name}\n")

def print_h5_tree(name, obj):
    indent = "  " * name.count("/")
    if isinstance(obj, h5py.Dataset):
        print(f"{indent}📄 {name}  shape={obj.shape}  dtype={obj.dtype}")
    else:
        print(f"{indent}📁 {name}/")

with h5py.File(demo_file, "r") as f:
    # 只看第一条 demo 的结构（全部 demo 结构相同，避免输出爆炸）
    print(f"顶层 keys: {list(f.keys())}")
    print(f"data 下的 demo 数量: {len(f['data'].keys())}")
    print(f"前 3 个 demo 名: {list(f['data'].keys())[:3]}\n")
    print("--- demo_0 的结构 ---")
    f["data/demo_0"].visititems(print_h5_tree)


## 3. 读出一条示教，看每一帧的形状

每条示教（demo）就是"人类操作机械臂完成一次任务"的完整轨迹：
- **actions**：每一帧机械臂执行的动作
- **obs/agentview_rgb**：俯视相机图像
- **obs/eye_in_hand_rgb**：腕部相机图像
- **obs/ee_*** / **obs/joint_states**：机器人状态


In [ ]:
with h5py.File(demo_file, "r") as f:
    demo = f["data/demo_0"]
    actions = demo["actions"][:]
    agent_imgs = demo["obs/agentview_rgb"][:]
    hand_imgs = demo["obs/eye_in_hand_rgb"][:]

print(f"轨迹总帧数 T = {len(actions)}")
print(f"actions       shape: {actions.shape}      dtype: {actions.dtype}")
print(f"agent 相机图像 shape: {agent_imgs.shape}  dtype: {agent_imgs.dtype}")
print(f"hand  相机图像 shape: {hand_imgs.shape}   dtype: {hand_imgs.dtype}")

print("\n动作向量 7 维含义（机械臂常见格式）:")
print("  [dx, dy, dz, droll, dpitch, dyaw, gripper]")
print(f"  第 0 帧动作: {actions[0]}")
print(f"  第 50 帧动作: {actions[min(50, len(actions)-1)]}")
print(f"\n动作范围: min={actions.min(axis=0)}\n         max={actions.max(axis=0)}")


## 4. 看几帧关键画面

把第一帧、中间帧、最后一帧画出来，肉眼确认机械臂确实在做事。


In [ ]:
T = len(agent_imgs)
indices = [0, T // 4, T // 2, 3 * T // 4, T - 1]

# LIBERO 的图像默认是上下颠倒的（mujoco 渲染惯例），翻一下才正常
def fix(img):
    return img[::-1]

fig, axes = plt.subplots(2, len(indices), figsize=(15, 6))
for col, idx in enumerate(indices):
    axes[0, col].imshow(fix(agent_imgs[idx]))
    axes[0, col].set_title(f"agent  t={idx}")
    axes[0, col].axis("off")
    axes[1, col].imshow(fix(hand_imgs[idx]))
    axes[1, col].set_title(f"hand   t={idx}")
    axes[1, col].axis("off")
plt.suptitle(f"Demo 0 of {demo_file.stem}", fontsize=10)
plt.tight_layout()
plt.show()


## 5. 把整条示教存成 GIF 动图

播放出来看机械臂动起来。GIF 会保存到 `outputs/demo0.gif`，可以在 Windows 资源管理器里直接打开看。


In [ ]:
import imageio

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# 拼接两个相机左右视图，每 2 帧抽一帧（减小体积）
frames = []
for i in range(0, len(agent_imgs), 2):
    a = fix(agent_imgs[i])
    h = fix(hand_imgs[i])
    frames.append(np.concatenate([a, h], axis=1))

gif_path = out_dir / "demo0.gif"
imageio.mimsave(gif_path, frames, duration=1 / 15, loop=0)
print(f"✅ GIF 已保存: {gif_path.resolve()}")
print(f"   共 {len(frames)} 帧, 大小约 {gif_path.stat().st_size/1024:.0f} KB")


## 6. 小结：我们学到了什么

把上面看到的总结一下，方便后面写训练代码时直接用。


In [ ]:
with h5py.File(demo_file, "r") as f:
    n_demos = len(f["data"].keys())
    lengths = [f[f"data/demo_{i}/actions"].shape[0] for i in range(n_demos)]

print(f"任务: {demo_file.stem}")
print(f"  示教条数: {n_demos}")
print(f"  轨迹长度: 最短 {min(lengths)} 帧, 最长 {max(lengths)} 帧, 平均 {np.mean(lengths):.0f} 帧")
print(f"  动作维度: 7 (xyz 平移 + rpy 旋转 + 1 维夹爪)")
print(f"  图像分辨率: 128 x 128 RGB, 两个相机 (agent / hand)")
print(f"\n下一步: 写一个 PyTorch Dataset 把这些数据喂给模型训练")
